In [1]:
using Healpix
using LinearAlgebra
using StaticArrays
using Base.Threads
using WignerD
using NPZ
using Plots
using Falcons
#using PyPlot
#using PyCall
using BenchmarkTools
using SparseArrays

In [2]:
include("../../BeamToyModel/src/BeamToyModel.jl")
include("../src/EBC.jl")

In [3]:
mutable struct ConvolutionCalculate{I<:Int, Bl<:Bool}
    nside_output::I
    lstart::I
    lstop::I
    mmax_calculate::I
    HWP::Bl
end

function ConvolutionCalculate(;
    nside_output::Int = 128,
    lstart::Int = 0,
    lstop::Int = 3*nside_output - 1,
    mmax_calculate::Int = 2,
    HWP::Bool = false
)
    lstart <= lstop || throw(ArgumentError("lstart must be <= lstop (got lstart=$lstart, lstop=$lstop)"))
    mmax_calculate <= lstop || throw(ArgumentError("mmax_calculate must be <= lstop (got mmax_calculate=$mmax_calculate, lstop=$lstop)"))
    return ConvolutionCalculate(
        nside_output,
        lstart,
        lstop,
        mmax_calculate,
        HWP
    )
end


ConvolutionCalculate

# WignerD

In [70]:
nside_in = 128
res = Resolution(nside_in)
lmax_in = 3nside_in-1
fwhm = deg2rad(1.0)
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = lmax_in)
;

In [71]:
cs = ConvolutionSky()
fieldnames(typeof(cs))
cb = ConvolutionBeam()
fieldnames(typeof(cb))
cc = ConvolutionCalculate(nside_output = nside_in, lstart = 0, mmax_calculate = 2)
fieldnames(typeof(cc))

(:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)

In [72]:
cb.lmax = lmax_in
#cb.mmax = 2
cb.blm = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm) 
;

In [73]:
nalm = Healpix.numberOfAlms(lmax_in, lmax_in)
almT = rand(ComplexF64, nalm)
almE = rand(ComplexF64, nalm)
almB = rand(ComplexF64, nalm)
cs.lmax = lmax_in
cs.alm = hcat(almT, almE, almB)
;

In [74]:
cc.lstop

383

In [75]:
n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
D_beam = spzeros(n_beam, n_beam)

n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
D_sky = spzeros(n_sky, n_beam)
;

In [76]:
function D_2ndtest(cb, cc, α, β, γ)
    n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
    n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    D_beam = spzeros(ComplexF64, n_sky, n_beam)
    for l in cc.lstart:cc.lstop
        #@show l
        #d_temp = WignerD.wignerd(ell,pi/2)
        n_ = min(l, cb.mmax)
        @inbounds for m in -l:l
            m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.lstop)
            #@show m_idx
            @inbounds for n in -n_:n_
                n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cb.mmax)
                D_beam[m_idx, n_idx] = WignerD.wignerDjmn(l, m, n, α, β, γ)
            end
        end
    end
    return D_beam
end

D_2ndtest (generic function with 1 method)

In [77]:
function D_1sttest(cb, cc, φ, θ, ψ)
    n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    D_beam = spzeros(ComplexF64, n_sky, n_sky)
    for l in cc.lstart:cc.lstop
        #@show l
        #d_temp = WignerD.wignerd(ell,pi/2)
        m_ = l
        @inbounds for m in -m_:m_
            m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.lstop)
            #@show m_idx
            @inbounds for n in -m_:m_
                n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cc.lstop)
                D_beam[m_idx, n_idx] = WignerD.wignerDjmn(l, m, n, φ, θ, ψ)
            end
        end
    end
    return D_beam
end

D_1sttest (generic function with 1 method)

In [78]:
θ = pi/3
φ = pi/4
dθ = rand()*1e-2
dφ = rand()*1e-2
ψ = rand()*2pi

1.964776059721558

In [79]:
θ = rand()*pi
φ = rand()*2pi
ψ = rand()*2pi
pix_ = ang2pixRing(res, θ, φ)
new_ang = pix2angRing(res, pix_)
err_, (alphas_, betas_, gammas_) = check_split(new_ang[2], new_ang[1], φ-new_ang[2], θ-new_ang[1], ψ)

(2.441102875394563e-14, (-1.0175655753713349, 0.0048161924832307796, 0.3695612851771046))

In [80]:
cb.mmax = 5
#n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)

5

In [ ]:
a = D_1sttest(cb, cc, new_ang[2], new_ang[1], 0)
b = D_2ndtest(cb, cc, alphas_, betas_, gammas_)
c=a*b

In [ ]:
d = D_2ndtest(cb, cc, φ, θ, ψ)

576×234 SparseMatrixCSC{ComplexF64, Int64} with 6226 stored entries:
⎡⠛⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠛⣷⣆⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠿⣇⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠛⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣇⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⣿⡇⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠿⣧⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⣿⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠿⣤⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢹⣶⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⎦

In [ ]:
maximum(abs.(c.-d))

1.3489394972766574e-14

In [ ]:
c[1:15,1:15]

15×15 SparseMatrixCSC{ComplexF64, Int64} with 71 stored entries:
 1.0-0.0im            ⋅            …             ⋅    
     ⋅      0.0394265+0.0680305im                ⋅    
     ⋅      -0.325027-0.198119im                 ⋅    
     ⋅       0.920253+0.0453727im                ⋅    
     ⋅                ⋅                          ⋅    
     ⋅                ⋅            …             ⋅    
     ⋅                ⋅                          ⋅    
     ⋅                ⋅                          ⋅    
     ⋅                ⋅                          ⋅    
     ⋅                ⋅                   0.5156+0.217759im
     ⋅                ⋅            …   -0.446246+0.0441112im
     ⋅                ⋅                -0.495697+0.33671im
     ⋅                ⋅                -0.153227+0.297236im
     ⋅                ⋅               0.00261723+0.118036im
     ⋅                ⋅                0.0139184+0.0242912im

In [ ]:
function local_effective_wignerD(cb, cc, α, β, γ)
    n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
    n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    D_beam = spzeros(ComplexF64, n_sky, n_beam)
    @inbounds for l in cc.lstart:cc.lstop
        n_ = min(l, cb.mmax)
        @inbounds for i in eachindex(α)
            @inbounds for m in -l:l
                m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.lstop)
                @inbounds for n in -n_:n_
                    n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cb.mmax)
                    D_beam[m_idx, n_idx] += WignerD.wignerDjmn(l, m, n, α[i], β[i], γ[i])
                end
            end
        end
    end
    return D_beam./length(α)
end

local_effective_wignerD (generic function with 1 method)

In [ ]:
nptg = 10
θ = pi/3
φ = pi/4
dθ = rand(nptg)*1e-2
dφ = rand(nptg)*1e-2
ψ = rand(nptg)*2pi

10-element Vector{Float64}:
 5.721530155416721
 4.0084227599312845
 2.3701617456497623
 5.450746134286755
 1.5438473363538479
 0.6746146373944563
 4.504880616855977
 5.279309004385452
 1.4211556430975218
 3.5699006015523933

In [ ]:
alphas = zeros(size(dθ))
betas = zeros(size(dθ))
gammas = zeros(size(dθ))

for i in eachindex(dθ)
    err, (alphas[i], betas[i], gammas[i]) = check_split(φ, θ, dφ[i], dθ[i], ψ[i])
    if err >= 1e0
        @show err
    end
end

In [ ]:
A = local_effective_wignerD(cb, cc, alphas, betas, gammas)
B = D_1sttest(cb, cc, φ, θ, 0)
C = B*A

576×234 SparseMatrixCSC{ComplexF64, Int64} with 6226 stored entries:
⎡⠛⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠛⣷⣆⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠿⣇⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠛⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣇⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⣿⡇⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠿⣧⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⣿⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠿⣤⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢹⣶⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⎦

In [ ]:
function D_beam_effective_direct(cb, cc, φ, θ, ψ)
    n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
    n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    D_beam = spzeros(ComplexF64, n_sky, n_beam)
    @inbounds for l in cc.lstart:cc.lstop
        n_ = min(l, cb.mmax)
        @inbounds for i in eachindex(φ)
            @inbounds for m in -l:l
                m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.lstop)
                @inbounds for n in -n_:n_
                    n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cb.mmax)
                    D_beam[m_idx, n_idx] += WignerD.wignerDjmn(l, m, n, φ[i], θ[i], ψ[i])
                end
            end
        end
    end
    return D_beam./length(φ)
end

D_beam_effective_direct (generic function with 1 method)

In [ ]:
D =  D_beam_effective_direct(cb, cc, φ.+dφ, θ.+dθ, ψ)


576×234 SparseMatrixCSC{ComplexF64, Int64} with 6226 stored entries:
⎡⠛⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠛⣷⣆⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠿⣇⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠛⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣇⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⣿⡇⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠿⣧⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⣿⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠿⣤⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢹⣶⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⎦

In [ ]:
maximum(abs.(C.-D))

7.870524519484338e-14

alphas = zeros(size(dθ))
betas = zeros(size(dθ))
gammas = zeros(size(dθ))

for i in eachindex(dθ)
    err, (alphas[i], betas[i], gammas[i]) = check_split(φ, θ, dφ[i], dθ[i], ψ[i])
    if err >= 1e0
        @show err
    end
end

alphas = zeros(size(dθ))
betas = zeros(size(dθ))
gammas = zeros(size(dθ))

for i in eachindex(dθ)
    err, (alphas[i], betas[i], gammas[i]) = check_split(φ, θ, dφ[i], dθ[i], ψ[i])
    if err >= 1e0
        @show err
    end
end

In [72]:
# first
# 1step version
function test1(ell, φ, θ, ψ)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    for i in eachindex(φ)
        D_result .+= WignerD.wignerD(ell, φ[i], θ[i], ψ[i])
    end
    return D_result./length(φ)
end

test1 (generic function with 1 method)

In [73]:
# second
# 2step version
function test2(ell, φ, θ, ψ, φ_pix, θ_pix)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    #d = WignerD.wignerd(ell, pi/2)
    D_1st = WignerD.wignerD(ell, φ_pix, θ_pix, 0.0)
    for i in eachindex(φ)
        dφ = φ[i] - φ_pix
        dθ = θ[i] - θ_pix
        err, (α, β, γ) = check_split(φ_pix, θ_pix, dφ, dθ, ψ[i])
        D_2nd = WignerD.wignerD(ell, α, β, γ)
        D_result .+=  D_1st * D_2nd
    end
    return D_result./length(φ)
end

test2 (generic function with 1 method)

In [78]:
ell = 2

2

In [79]:
D1 = test1(ell, φ.+dφ, θ.+dθ, ψ)

5×5 Matrix{ComplexF64}:
  -0.634771+0.342097im    -0.499621+0.346139im   …  -0.0134069+0.0183805im
  0.0821529+0.602232im   -0.0824054-0.326603im       -0.059335-0.0901947im
   0.305957+0.0694306im    -0.60838-0.0681631im       0.305957-0.0694306im
   0.059335-0.0901947im   -0.163814+0.322249im      -0.0821529+0.602232im
 -0.0134069-0.0183805im    0.072938+0.0795976im      -0.634771-0.342097im

In [80]:
D2 = test2(ell, φ.+dφ, θ.+dθ, ψ, φ, θ)

5×5 Matrix{ComplexF64}:
  -0.634771+0.342097im    -0.499621+0.346139im   …  -0.0134069+0.0183805im
  0.0821529+0.602232im   -0.0824054-0.326603im       -0.059335-0.0901947im
   0.305957+0.0694306im    -0.60838-0.0681631im       0.305957-0.0694306im
   0.059335-0.0901947im   -0.163814+0.322249im      -0.0821529+0.602232im
 -0.0134069-0.0183805im    0.072938+0.0795976im      -0.634771-0.342097im

In [81]:
maximum(abs.(D1 .- D2))

2.3148150063483114e-14

In [ ]:
d = D_1sttest(cb, cc, φ+dφ, θ+dθ, ψ)

In [69]:
(c.-d)[1:10,1:10]

10×10 SparseMatrixCSC{ComplexF64, Int64} with 35 stored entries:
     ⋅                 ⋅             …            ⋅    
     ⋅      -0.0221298+0.0254459im                ⋅    
     ⋅      -0.0246111-0.00160399im               ⋅    
     ⋅       -0.012874+0.014462im                 ⋅    
     ⋅                 ⋅                          ⋅    
     ⋅                 ⋅             …            ⋅    
     ⋅                 ⋅                          ⋅    
     ⋅                 ⋅                          ⋅    
     ⋅                 ⋅                          ⋅    
     ⋅                 ⋅                0.0133757-0.070056im

In [70]:
c[1:10,1:10]

10×10 SparseMatrixCSC{ComplexF64, Int64} with 36 stored entries:
 1.0-0.0im             ⋅           …           ⋅    
     ⋅       -0.229873-0.797918im              ⋅    
     ⋅       -0.527591-0.057958im              ⋅    
     ⋅      -0.0812246+0.148919im              ⋅    
     ⋅                 ⋅                       ⋅    
     ⋅                 ⋅           …           ⋅    
     ⋅                 ⋅                       ⋅    
     ⋅                 ⋅                       ⋅    
     ⋅                 ⋅                       ⋅    
     ⋅                 ⋅              0.426915+0.381523im

In [71]:
d[1:10,1:10]

10×10 SparseMatrixCSC{ComplexF64, Int64} with 36 stored entries:
 1.0-0.0im             ⋅           …           ⋅    
     ⋅       -0.207743-0.823364im              ⋅    
     ⋅        -0.50298-0.056354im              ⋅    
     ⋅      -0.0683507+0.134457im              ⋅    
     ⋅                 ⋅                       ⋅    
     ⋅                 ⋅           …           ⋅    
     ⋅                 ⋅                       ⋅    
     ⋅                 ⋅                       ⋅    
     ⋅                 ⋅                       ⋅    
     ⋅                 ⋅              0.413539+0.451579im